In [3]:
import pandas as pd
import ast

# Đọc file gốc
df = pd.read_csv('courses_cleaned.csv')
df.insert(0, 'external_course_id', range(1, len(df) + 1))

# Hàm parse cột skills (đang là string dạng list, VD: "['Python', 'SQL']")
def parse_skills(s):
    try:
        lst = ast.literal_eval(s)   # chuyển string thành list Python thật
        if isinstance(lst, list):
            return [str(x).strip() for x in lst if str(x).strip()]
    except Exception:
        pass
    return []

df['skills_parsed'] = df['course_skills'].apply(parse_skills)

# Tạo bảng skills (danh sách kỹ năng duy nhất, không trùng lặp)
all_skills = sorted({sk for row in df['skills_parsed'] for sk in row})

# Dọn lỗi ký tự "-t" bị dính ở đầu 1 số skill (do lỗi crawl dữ liệu gốc)
all_skills_cleaned = [sk[2:] if sk.startswith('-t') else sk for sk in all_skills]

skills_df = pd.DataFrame({
    'skill_id': range(1, len(all_skills_cleaned) + 1),
    'ten_ky_nang': all_skills_cleaned
})
skill_to_id = dict(zip(all_skills, skills_df['skill_id']))  # map theo tên GỐC (chưa dọn) để khớp key

# Tạo bảng trung gian (khóa học nào - có skill nào)
rows = []
for _, row in df.iterrows():
    for sk in row['skills_parsed']:
        rows.append({'external_course_id': row['external_course_id'], 'skill_id': skill_to_id[sk]})
junction_df = pd.DataFrame(rows)

# Tạo bảng external_courses (bỏ cột skills gốc vì đã tách riêng)
ext_courses_df = df.drop(columns=['course_skills', 'skills_parsed']).rename(columns={
    'course_title': 'title',
    'course_organization': 'organization',
    'course_certificate_type': 'certificate_type',
    'course_time': 'duration',
    'course_rating': 'rating',
    'course_reviews_num': 'reviews_num',
    'course_difficulty': 'difficulty',
    'course_url': 'external_url',
    'course_students_enrolled': 'students_enrolled',
})

# Xuất ra 3 file CSV
ext_courses_df.to_csv('external_courses.csv', index=False)
skills_df.to_csv('skills.csv', index=False)
junction_df.to_csv('external_course_skills.csv', index=False)

print(f'Số khóa học: {len(ext_courses_df)}')
print(f'Số kỹ năng duy nhất: {len(skills_df)}')
print(f'Số dòng bảng trung gian: {len(junction_df)}')

Số khóa học: 809
Số kỹ năng duy nhất: 2237
Số dòng bảng trung gian: 4482
